In [ ]:
""" VERİ SETİNİ YÜKLEME """
import os
import polars as pl
import numpy as np

# Veriyi yükle
DATA_PATH = "/kaggle/input/datasets/denizbyat/metropt3-features/metropt3_features.parquet"

df = pl.read_parquet(DATA_PATH)

print(f"Satır sayısı : {df.shape[0]:,}")
print(f"Sütun sayısı : {df.shape[1]}")
print(f"RAM kullanımı: {df.estimated_size('mb'):.1f} MB")

Satır sayısı : 1,508,308 -- 
Sütun sayısı : 133 -- 
RAM kullanımı: 758.1 MB

In [ ]:
""" HAZIRLIK VE NORMALİZSAYON """
import numpy as np
from sklearn.preprocessing import StandardScaler

# timestamp ve is_suspect'i ayır
feature_cols = [col for col in df.columns 
                if col not in ['timestamp', 'is_suspect']]      # timestampi saten ayrı ayrı özellik olarak verdik, is_suspect ise hedef değişkenimiz

X = df.select(feature_cols).to_numpy()  # girdi (verdiğimiz özelliklerden çıkarım yapacak)
y = df['is_suspect'].to_numpy()     # cevap (doğru mu diye bakacak)

print(f"Feature matrix : {X.shape}")
print(f"Normal kayıt   : {(y == 0).sum():,} (%{(y == 0).mean()*100:.1f})")
print(f"Şüpheli kayıt  : {(y == 1).sum():,} (%{(y == 1).mean()*100:.1f})")

# Normalize ediyoruz çünkü veri setindeki özellikler farklı ölçeklerde olabilir ve bu, modelin öğrenmesini zorlaştırabilir.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nNormalizasyon tamamlandı")

Feature matrix : (1508308, 131) -- 
Normal kayıt   : 1,406,713 (%93.3) -- 
Şüpheli kayıt  : 101,595 (%6.7)

In [ ]:
""" ISOLATION FOREST MODELİ """
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(
    n_estimators=100,      # 100 karar ağacı
    contamination=0.067,   # Verimizdeki anomali oranı (%6.7) üstte belirttiğimiz gibi
    random_state=42,
    n_jobs=-1              # Tüm CPU çekirdeklerini kullan
)

print("Isolation Forest eğitiliyor")
iso_forest.fit(X_scaled)

# Tahmin — 1: normal, -1: anomali
iso_predictions = iso_forest.predict(X_scaled)
iso_predictions = (iso_predictions == -1).astype(int)  # -1 → 1 (anomali)

print("Eğitim tamamlandı ✓")
print(f"Anomali tespit edilen kayıt: {iso_predictions.sum():,} (%{iso_predictions.mean()*100:.1f})")

aşağıdaki Değerlendirme hücresine bakmamız gerekiyor.

isolation forest modelinin mantığı normal ve anomali durumlarının kümelenmesi mantığına dayanır. Mesela ormanda bulunan farklı toplumları düşünün. Bulunduğunuz topluluk ne kadar küçükse sizin adımlar atarak toplumunuzdan uzaklaşıp kaybolmanızda daha büyük bir toplumdan uzaklaşmanıza göre daha az sayıda adım gerektirir. Prensip temelde buna dayanır. Şüpheli durumlar küçük topluluk, Normal durumlar ise büyük toplumu ifade edeceğinden (çünkü anomali durumlar azınlıktadır) model de ona kontrol etmesi için verdiğimiz veriyi deneyerek ne kadar adımda kaybolduğuna bakarak bunu tespit eder.

In [ ]:
""" DEĞERLENDİRME """
from sklearn.metrics import classification_report, confusion_matrix

# precision, recall, f1-score, support metriklerini gösteren rapordur
print("=== ISOLATION FOREST SONUÇLARI ===\n")
print(classification_report(y, iso_predictions, 
                            target_names=['Normal', 'Şüpheli']))

# Gerçek ve tahmin edilen sınıfların karşılaştırıldığı tablodur
print("\nKarmaşıklık Matrisi:")
cm = confusion_matrix(y, iso_predictions)
print(f"                 Tahmin Normal  Tahmin Şüpheli")
print(f"Gerçek Normal  : {cm[0][0]:>12,}  {cm[0][1]:>14,}")
print(f"Gerçek Şüpheli : {cm[1][0]:>12,}  {cm[1][1]:>14,}")

Sonuçlar aşırı zayıf çıktı. örneğin f1: 0.08 değerinde. karmaşıklık matriside aynı şekilde iyi değil. şüpheli durumların %8 civarını yakalayabilmiş. Bunun sebebi muhtemelen veri setimizin gürültüsünün fazla olması. Sistem durmasından önce 2 saatlik pencereleri baz aldık ancak bunların çoğu bilerek bakım tarzında kapatmalar olabiliyor. Model doğru tahmin yapıyor ancak set fazla gürültülü olduğundan hata çok.

In [ ]:
""" OneClassSVM MODELİ """
from sklearn.svm import OneClassSVM

# One-Class SVM yi 1.5M satır çok fazla olduğundan örneklem alıp eğiticez
sample_size = 100_000   # Normal kayıtların %6.7'si kadar örneklem alarak dengeli ve rastgele bir eğitim seti oluşturucaz
normal_idx = np.where(y == 0)[0]    # normal kayıtların indekslerini buluyoruz
sample_idx = np.random.choice(normal_idx, size=sample_size, replace=False)  # normal indexler içinden örneklem için rastgele seçiyoruz
X_normal_sample = X_scaled[sample_idx]      # seçilen indekslere göre normal kayıtların özelliklerini alıyoruz

print(f"Eğitim örneklemi: {X_normal_sample.shape[0]:,} normal kayıt")

ocsvm = OneClassSVM(
    kernel='rbf',   # RBF çekirdeği seçiyoruz. verilerin doğrusal olmayan bir şekilde ayrılmasına olanak tanıyacak yani düz çizgilerle değilde daha esnek daha doğru sonuç verecektir. Sensör verileri gibi kesin bir sınır olmayan verilerde RBF çekirdeği genellikle iyi sonuç verir
    nu=0.067,      # Anomali oranımız (bizim veri setimizdeki şüpheli kayıtların oranı) kadar anomali beklediğimizi belirtiyoruz
    gamma='scale'   # gamma parametresi, RBF çekirdeği için önemli bir parametredir. 'scale' değeri, veri setinin özellik sayısına göre otomatik olarak gamma değerini (sınır esnekliğini) ayarlar demektir
)

print("One-Class SVM eğitiliyor...")
ocsvm.fit(X_normal_sample)  # sadece normal kayıtlarla eğitiyoruz ki normalin nasıl olduğunu öğrenebilsin ve ona göre anomali tespit edebilsin

# Tüm veri üzerinde tahmin
print("Tahmin yapılıyor...")
ocsvm_predictions = ocsvm.predict(X_scaled)     # tüm veriler ile tahmin = -1 → anomali, 1 → normal olacak
ocsvm_predictions = (ocsvm_predictions == -1).astype(int)   # yukarıda sonuç -1 ve 1 olarak geldiği için -1 olanları True yapıyoruz ve sonra int'e çeviriyoruz yani -1 → 1 (anomali), 1 → 0 (normal) yapıyoruz

print("Tamamlandı ✓")
print(f"Anomali tespit edilen: {ocsvm_predictions.sum():,} (%{ocsvm_predictions.mean()*100:.1f})")

Oneclassın kabaca mantığı normal olanların etrafına sınır çizip sınır dışındakileri anomali ilan etmektir. Normal sensörlerin davranışlarını öğreniyor belirlediği aralıklara (sürekli tekrar eden) sınırları çiziyor ve diğer değerleri anomali ilan ediyor. isolation foreste göre yavaştır ancak daha güçlüdür. Çünkü isolation anomaliyi analiz ederken onaclass normal değerlerden çıkarım yaparak tespit etmeye çalışır.

In [ ]:
""" DEĞERLENDİRME """
print("=== ONE-CLASS SVM SONUÇLARI ===\n")
print(classification_report(y, ocsvm_predictions,
                            target_names=['Normal', 'Şüpheli']))

print("\nKarmaşıklık Matrisi:")
cm_ocsvm = confusion_matrix(y, ocsvm_predictions)
print(f"                 Tahmin Normal  Tahmin Şüpheli")
print(f"Gerçek Normal  : {cm_ocsvm[0][0]:>12,}  {cm_ocsvm[0][1]:>14,}")
print(f"Gerçek Şüpheli : {cm_ocsvm[1][0]:>12,}  {cm_ocsvm[1][1]:>14,}")

Sonuç forest modeli ile neredeyse birebir aynı yine f1 skoru başta olmaz üzere aşırı kötü sonuçlar. Bunun sebebi öncekinde de söylediğimz gibi etiketlerimizde. Tahminler belki doğrudur ancak gürültü çok fazla olduğundan bunun ölçümünü yapamıyoruz

In [ ]:
""" LSTM İÇİN VERİ HAZIRLAMA """
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector, TimeDistributed
from tensorflow.keras.callbacks import EarlyStopping

# Sadece normal olan veriyle eğiteceğiz
X_normal = X_scaled[y == 0]

# LSTM için 3D shape gerekiyor (samples, timesteps, features) -- Her örneği 10 adımlık (10 adım = 10 x 10sn = 100sn) pencere olarak düzenle
timesteps = 10      # 10 adımlık zaman penceresi kullanarak modelin kısa süreli bağımlılıkları öğrenmesini sağlıyoruz
n_features = X_scaled.shape[1]      # özellik sayısı (sütun sayımız)

# Bu fonksiyonla pencereleri kaydırma işlemi  yapıyoruz ve her pencere için özellikleri düzenliyoruz
def create_sequences(data, timesteps):
    sequences = []
    for i in range(len(data) - timesteps):      # veri uzunluğundan timesteps kadar eksik olacak şekilde döngü kuruyoruz çünkü her pencere timesteps kadar veri içeriyor
        sequences.append(data[i:i+timesteps])   # örneğin 0-9, 1-10, 2-11 şeklinde kaydırarak pencereler oluşturuyoruz
    return np.array(sequences)

# Eğitim için örneklem al — tüm veri çok büyük
sample_size = 50_000
sample_idx = np.random.choice(len(X_normal), size=sample_size, replace=False)   # normal kayıtlar içinden rastgele örneklem alıyoruz (false ile tekrarı önlüyoruz)
X_train_sample = X_normal[sample_idx]
X_train_seq = create_sequences(X_train_sample, timesteps)

print(f"Eğitim verisi shape: {X_train_seq.shape}")

LSTM modeli 3 boyutlu zaman yapısı istediğinden ona göre verileri düzenliyoruz. 3 Boyutlu istemesinin sebebi diğer modellere göre farklı bir öğrenme mekanizması olması (sonraki hücrelerde açıklıyorum). Kabaca bu model diğer 2 modele göre en büyük farkı 3 boyutlu çalışmasıdır. Yani ekstra zaman dizesinide aktif olarak kullanır. Diğerleri önceki kadar verdiğimiz zamana bakarak tespit eder bu model ise komple inceleyerek anomaliyi bulur. Örnekle Boyut 1 — Kaç hasta?: 49,990 hasta -- Boyut 2 — Kaç saniye EKG?: 10 saniye -- Boyut 3 — Kaç ölçüm/saniye?: 131 sensör. her hastanın 10 saniyelik 131 kanallı EKG'sine bakıyor. Tek bir anlık fotoğrafa değil, zaman içindeki akışa bakıyor.

In [ ]:
# LSTM Autoencoder mimarisi
inputs = Input(shape=(timesteps, n_features))   # girdi şekli (10 adımlık pencere, özellik sayısı)

# Encoder (veriyi sıkıltıyoruz)
encoded = LSTM(64, activation='relu', return_sequences=False)(inputs)   # 64 boyutlu gizli katman, return_sequences=False ile sadece son çıktıyı alıyoruz çünkü bu katman veriyi tek bir vektöre sıkıştıracak

# Decoder için tekrarla
repeated = RepeatVector(timesteps)(encoded) #   encoded vektörünü timesteps kadar tekrarlayarak decoder için uygun hale getiriyoruz (10 adımlık pencere oluşturuyoruz)

# Decoder
decoded = LSTM(64, activation='relu', return_sequences=True)(repeated)  # decoder katmanı, return_sequences=True ile her zaman adımı için çıktı üretiyoruz çünkü her adımda özellikleri yeniden oluşturacağız
outputs = TimeDistributed(Dense(n_features))(decoded)

# Model
autoencoder = Model(inputs, outputs)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.summary()

Model veriyi sıkıştırıp geri açıyor — girdiyi olduğu gibi çıktıya yansıtmayı öğreniyor. Normal veriyle eğitilince normal akışı iyi yeniden üretir, anomaliyi üretemez.

Katmanlar
- **Input:** 10 zaman adımı × 131 özellik
- **LSTM Encoder:** 1310 boyutu 64'e sıkıştırır — son adımın özeti alınır (`return_sequences=False`)
- **RepeatVector:** 64 boyutlu özeti 10 kez tekrarlar — Decoder'a zaman boyutu geri eklenir
- **LSTM Decoder:** Sıkıştırılmış veriyi açar (`return_sequences=True` — her adım çıkar)
- **TimeDistributed Dense:** Her zaman adımını 64'ten 131'e genişletir


In [ ]:
# Eğitim
early_stopping = EarlyStopping(
    monitor='val_loss',     # validation loss adım adım izlemek için
    patience=3,     # 3 epoch boyunca iyileşme olmazsa durdur
    restore_best_weights=True   # en iyi ağırlıkları geri yükle (validation loss en düşük olduğu epochun ağırlıklarını kullanarak modelin en iyi performansını sağlamış oluruz
)

history = autoencoder.fit(     
    X_train_seq, X_train_seq,  # Girdi = Çıktı (autoencoder)
    epochs=20,      # 20 epoch boyunca eğitiyoruz, erken durdurma ile daha az da sürebilir
    batch_size=512,        # büyük veri seti için uygun bir batch size seçiyoruz
    validation_split=0.1,   # verinin %10'unu doğrulama için ayırıyoruz
    callbacks=[early_stopping], # erken durdurma callback'ini ekliyoruz
    verbose=1       # eğitim sürecini detaylı göstermek için verbose=1 kullanıyoruz
)

print("Eğitim tamamlandı ✓")

20 epoch'un tamamı çalıştı — EarlyStopping devreye girmedi. Train loss: 0.98 → 0.58 (sürekli düştü). Val loss: 0.87 → 0.56 (sürekli düştü). Train ve val_loss birbirine yakın overfit yok. 20 epoch yetmedi model hâlâ öğreniyordu. Daha fazla epoch verilseydi loss daha da düşebilirdi. şimdi model 50bin normal sensör kaydı ile gerçek verilerin nasıl olduğunu öğrendi. Şimdi test ederek ona tüm verileri verip nasıl iş çıkaracağına bakabilirz. Kurulum aşamasında dediğim gibi 3D kontrol yapacak yani sadece o anki değerlere değil geçimişten gelene kadar ki değerleri inceleyecek.